# Этап 1

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Импорт библиотек

In [26]:
import numpy as np
from matplotlib import pyplot as plt
import pandas as pd
from scipy.signal import savgol_filter
from scipy import fft
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

Считывание данных из подготовленного файла

In [27]:
data = pd.read_csv('data.csv', header=None, names=['t', 'a', 'klass'])

Считывание данных для обучение модели

In [28]:
train_data = pd.read_csv('train_data.csv', names=['t', 'a', 'klass'])

In [38]:
def extract_features(segment):
    """Извлекает признаки из спектра Фурье."""
    N = len(segment)
    features = {}
    
    raw_centered = segment - np.mean(segment)
    yf_raw = fft.fft(raw_centered)
    spectrum_raw = np.abs(yf_raw[:N // 2 + 1])
    harmonics = np.arange(N // 2 + 1)
    max_harmonic = N // 2
    
    low_threshold = max(1, int(max_harmonic * 0.1))
    mid_threshold = int(max_harmonic * 0.4)
    
    low_mask = (harmonics >= 1) & (harmonics < low_threshold)
    high_mask = harmonics >= mid_threshold
    
    low_energy = np.sum(spectrum_raw[low_mask] ** 2)
    high_energy = np.sum(spectrum_raw[high_mask] ** 2)
    total_energy = np.sum(spectrum_raw ** 2)
    
    features['high_to_low_ratio'] = high_energy / (low_energy + 1e-10)
    
    # Спектральный наклон
    log_spectrum = np.log(spectrum_raw[1:] + 1e-10)
    log_harmonics = np.log(harmonics[1:] + 1e-10)
    if len(log_harmonics) > 1:
        features['spectral_slope'] = np.polyfit(log_harmonics, log_spectrum, 1)[0]
    else:
        features['spectral_slope'] = 0
    
    # Спектральная энтропия
    spectrum_norm = spectrum_raw[1:] / (np.sum(spectrum_raw[1:]) + 1e-10)
    features['spectral_entropy'] = -np.sum(
        spectrum_norm * np.log(spectrum_norm)
    )
    
    # Доля энергии на самых низких частотах
    very_low_mask = (harmonics >= 1) & (harmonics < max(1, int(max_harmonic * 0.05)))
    features['dc_dominance'] = (
        np.sum(spectrum_raw[very_low_mask] ** 2) / (total_energy)
    )
    
    if N >= 21:
        envelope = savgol_filter(segment, window_length=21, polyorder=3)
    else:
        envelope = segment.copy()
    
    # Наклон огибающей
    t = np.arange(N)
    features['envelope_slope'] = np.polyfit(t, envelope, 1)[0]
    
    return features

Подготовка тренировочных данных

In [39]:
SEGMENT_LENGTH = 100
train_segments = []
train_labels = []

for i in range(0, len(train_data) - SEGMENT_LENGTH + 1, SEGMENT_LENGTH):
    segment = train_data['a'].iloc[i:i+SEGMENT_LENGTH].values
    label = train_data['klass'].iloc[i]
    train_segments.append(segment)
    train_labels.append(label)

X = np.array([list(extract_features(seg).values()) for seg in train_segments])
y = np.array(train_labels)

C:\Users\ilyak\AppData\Local\Temp\ipykernel_17044\248206117.py:35: RuntimeWarning: divide by zero encountered in log
  spectrum_norm * np.log(spectrum_norm)
C:\Users\ilyak\AppData\Local\Temp\ipykernel_17044\248206117.py:35: RuntimeWarning: invalid value encountered in multiply
  spectrum_norm * np.log(spectrum_norm)


Подготовка данных для предсказания

In [40]:
TEST_SEGMENT_LENGTH = 20
test_segments = []
test_labels = []

for i in range(0, len(data) - TEST_SEGMENT_LENGTH + 1, TEST_SEGMENT_LENGTH):
    segment = data['a'].iloc[i:i+TEST_SEGMENT_LENGTH].values
    label = data['klass'].iloc[i]
    test_segments.append(segment)
    test_labels.append(label)

X_test = np.array([list(extract_features(seg).values()) for seg in test_segments])
y_test = np.array(test_labels)

C:\Users\ilyak\AppData\Local\Temp\ipykernel_17044\248206117.py:35: RuntimeWarning: divide by zero encountered in log
  spectrum_norm * np.log(spectrum_norm)
C:\Users\ilyak\AppData\Local\Temp\ipykernel_17044\248206117.py:35: RuntimeWarning: invalid value encountered in multiply
  spectrum_norm * np.log(spectrum_norm)


In [41]:
X_test

array([[ 2.05515166e+14, -1.95907671e-01,  2.14998879e+00,
         0.00000000e+00,  8.90977444e-01],
       [ 3.90092319e+14,  6.37561857e-01,  2.10089226e+00,
         0.00000000e+00, -2.48872180e-01],
       [ 4.64422720e+14, -1.39110851e-01,  2.17457095e+00,
         0.00000000e+00,  6.14285714e-01],
       [ 1.94838342e+14, -1.64057504e-02,  2.13177333e+00,
         0.00000000e+00, -7.33834586e-01],
       [ 2.38989127e+14, -7.54434313e-02,  2.08501745e+00,
         0.00000000e+00, -4.30827068e-01],
       [ 3.35888917e+14, -1.09341687e-01,  2.12093865e+00,
         0.00000000e+00,  1.72932331e+00],
       [ 3.08077954e+14, -1.76447217e-01,  2.24922484e+00,
         0.00000000e+00,  1.51879699e+00],
       [ 5.75562167e+14,  1.56287360e-01,  2.19128472e+00,
         0.00000000e+00,  1.31804511e+00],
       [ 1.79598469e+14, -4.19417247e-01,  2.21040714e+00,
         0.00000000e+00,  4.96992481e-01],
       [ 3.63327529e+14, -4.62839545e-01,  2.17800040e+00,
         0.00000000e+00

Обучение модели

In [42]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

model = RandomForestClassifier(n_estimators=200, random_state=42, max_depth=10)
model.fit(X_scaled, y)

y_pred = model.predict(X_test)
accuracy = np.mean(y_pred == y_test)

In [43]:
accuracy

np.float64(0.5)

In [34]:
y_pred

array([1, 1, 1, 1, 1, 1, 1, 1, 1, 1])